# 🛡️ Fraud Detection - PaySim1 Dataset

Ce notebook couvre l'ensemble du pipeline de détection de fraude.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Chargement des Données

In [ ]:
# Charger le dataset PaySim1
# Téléchargez depuis: https://www.kaggle.com/datasets/ealaxi/paysim1
df = pd.read_csv('data/raw/PS_20174392719_1491204439457_log.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Fraud rate: {df.isFraud.mean()*100:.3f}%')

## 2. Analyse Exploratoire

In [ ]:
# Distribution des types
df['type'].value_counts().plot(kind='bar')
plt.title('Distribution des Types de Transactions')
plt.show()

# Fraude par type
fraud_by_type = df.groupby('type')['isFraud'].agg(['count', 'sum', 'mean'])
print(fraud_by_type)

## 3. Feature Engineering

In [ ]:
from src.feature_engineering import FeatureEngineer

fe = FeatureEngineer()
df_features = fe.fit_transform(df)

feature_cols = fe.get_feature_names()
X = df_features[feature_cols].fillna(0)
y = df['isFraud']

print(f'Features créées: {len(feature_cols)}')

## 4. Modèles

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Isolation Forest
iso = IsolationForest(n_estimators=200, contamination=0.002, random_state=42)
iso.fit(X_train_scaled)

iso_scores = iso.decision_function(X_test_scaled)
iso_preds = (iso_scores < np.percentile(iso_scores, 2)).astype(int)

print(f'AUC-ROC: {roc_auc_score(y_test, -iso_scores):.4f}')
print(classification_report(y_test, iso_preds))

In [ ]:
# Autoencoder
from sklearn.neural_network import MLPRegressor

X_train_normal = X_train_scaled[y_train == 0]

auto = MLPRegressor(hidden_layer_sizes=(32, 16, 32), max_iter=200, random_state=42)
auto.fit(X_train_normal, X_train_normal)

X_pred = auto.predict(X_test_scaled)
recon_error = np.mean((X_test_scaled - X_pred)**2, axis=1)

print(f'AUC-ROC: {roc_auc_score(y_test, recon_error):.4f}')

## 5. Sauvegarde

In [ ]:
import joblib

joblib.dump(iso, 'models/isolation_forest.joblib')
joblib.dump(auto, 'models/autoencoder.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

print('✅ Modèles sauvegardés')